# Experimento do TCC — Text-to-SQL com recuperação de demonstraçõesEste notebook não contém lógica do experimento. Ele clona o repositório,instala, escreve a configuração e chama a linha de comando — a lógica todavive no GitHub, versionada.**Ordem de uso:** rode as células 1 a 5 **sem GPU** (teste de fumaça). Sódepois que elas passarem, ligue a T4 e siga para a parte 2.

## 1 · Montar o Drive e descompactar os dados

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Só na primeira sessão. Depois pode comentar estas duas linhas.
!unzip -q -o /content/drive/MyDrive/tcc/spider_tcc.zip -d /content/drive/MyDrive/tcc/

!ls /content/drive/MyDrive/tcc/data/spider/

A listagem tem que mostrar `tables.json`, `train_spider.json`, `dev.json` e `database`.

## 2 · Trazer o código

In [ ]:
%cd /content
!rm -rf TCC2
!git clone -q https://github.com/VitorEduardoLimaKenor/TCC2.git
%cd TCC2
!git log --oneline -1

O `rm -rf` é intencional: toda sessão começa do código que está no GitHub. Você nunca edita código aqui dentro — edita na sua máquina, dá `push`, e roda esta célula de novo.

## 3 · Instalar

In [ ]:
!pip install -q -e .
!python -c "import tcc_rag; print('tcc_rag ok')"

## 4 · Configuração do teste de fumaça`backend: dummy` não carrega modelo nenhum. Esta rodada não mede nada — ela responde a outra pergunta, mais urgente: o encanamento está ligado?

In [ ]:
%%writefile configs/colab.yaml
name: colab-fumaca

paths:
  spider_root: /content/drive/MyDrive/tcc/data/spider
  questions_pt: /content/drive/MyDrive/tcc/data/spider_pt/questions_pt.json
  work_dir: /content/drive/MyDrive/tcc/artifacts

data:
  min_questions_per_db: 90
  min_families_per_db: 25
  pool_ratio: 0.60
  val_ratio: 0.15
  test_ratio: 0.25
  near_duplicate_threshold: 0.92
  seed: 20260904

embedding:
  model: intfloat/multilingual-e5-base
  device: cpu
  batch_size: 64
  normalize: true
  query_prefix: "query: "
  passage_prefix: "passage: "

qdrant:
  mode: local
  path: /content/drive/MyDrive/tcc/artifacts/qdrant
  collection: demos
  distance: Cosine

retrieval:
  k: 4
  n_candidates: 25
  lambda_question: 0.5
  fixed_demo_seed: 20260919

generation:
  backend: dummy
  model: none
  dtype: float16
  max_context_tokens: 4096
  max_new_tokens: 256
  seed: 20261010
  batch_size: 8

sandbox:
  timeout_s: 15.0
  max_rows: 5000

## 5 · Rodar o pipeline completo, sem GPUA indexação baixa o modelo de *embeddings* (~1 GB) e vetoriza 2242 demonstrações na CPU. Leve alguns minutos.

In [ ]:
!tcc-rag prepare   --config configs/colab.yaml
!tcc-rag partition --config configs/colab.yaml
!tcc-rag leakage   --config configs/colab.yaml

In [ ]:
!tcc-rag index --config configs/colab.yaml

In [ ]:
!tcc-rag run      --config configs/colab.yaml --split val --limit 20
!tcc-rag evaluate --config configs/colab.yaml

### O que confirma que passou- `bancos elegíveis (>= 90 perguntas, >= 25 famílias): 17`- `VEREDITO: limpo`- `indexadas 2242 demonstrações`- cinco linhas `C0:` a `C4:`, cada uma com `+20 linhas`- a tabela final com a coluna `n` preenchida**EX igual a zero é o esperado aqui.** O gerador falso devolve `SELECT 1`.O que importa é que nada quebrou e que o log foi escrito no Drive.

---# Parte 2 · Com GPUAgora sim: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.A sessão reinicia. Rode de novo as células 1, 2 e 3, e depois siga daqui.

## 6 · Confirmar a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Precisa dizer `Tesla T4, 15360 MiB`. Se disser que não há GPU, a cota do Colab gratuito acabou — espere algumas horas ou use o Kaggle.

## 7 · Instalar o suporte a modelo e reescrever a configuração

In [ ]:
!pip install -q -e ".[hf]"
!pip install -q bitsandbytes

In [ ]:
%%writefile configs/colab.yaml
name: colab-gpu

paths:
  spider_root: /content/drive/MyDrive/tcc/data/spider
  questions_pt: /content/drive/MyDrive/tcc/data/spider_pt/questions_pt.json
  work_dir: /content/drive/MyDrive/tcc/artifacts

data:
  min_questions_per_db: 90
  min_families_per_db: 25
  pool_ratio: 0.60
  val_ratio: 0.15
  test_ratio: 0.25
  near_duplicate_threshold: 0.92
  seed: 20260904

embedding:
  model: intfloat/multilingual-e5-base
  device: cuda
  batch_size: 128
  normalize: true
  query_prefix: "query: "
  passage_prefix: "passage: "

qdrant:
  mode: local
  path: /content/drive/MyDrive/tcc/artifacts/qdrant
  collection: demos
  distance: Cosine

retrieval:
  k: 4
  n_candidates: 25
  lambda_question: 0.5
  fixed_demo_seed: 20260919

generation:
  backend: hf
  model: Qwen/Qwen2.5-Coder-7B-Instruct
  quantization: bitsandbytes-4bit
  dtype: float16
  max_context_tokens: 4096
  max_new_tokens: 256
  seed: 20261010
  batch_size: 4

sandbox:
  timeout_s: 15.0
  max_rows: 5000

## 8 · A rodada de calibraçãoEste é o passo que separa *"meu número é baixo"* de *"meu código estáquebrado"*, e ele vem antes de qualquer construção em cima.Roda C0 (sem demonstrações) em inglês. O primeiro comando baixa o modelo(~5 GB) — leva alguns minutos na primeira vez.

In [ ]:
!tcc-rag run --config configs/colab.yaml --split val --limit 80 \
             --conditions C0 --langs en
!tcc-rag evaluate --config configs/colab.yaml

### Como ler o resultado| EX de C0 em inglês | Leitura ||---|---|| **0,45 – 0,65** | Coerente. Um 7B em 4 bits abaixo de um 14B é esperado. Siga. || **0,25 – 0,45** | Baixo demais para ser só o porte. Quase sempre é a extração do SQL da resposta, ou o modelo respondendo com explicação em vez de consulta. || **abaixo de 0,20** | Algo quebrado. Suspeitos, nesta ordem: o DDL não chega ao *prompt*, o avaliador compara errado, ou o caminho do SQLite está errado e toda execução falha. |Carvalho *et al.* reportaram ~0,63 para o Qwen2.5-Coder-14B em inglês.

## 9 · Investigar, se o número não bater

In [ ]:
import json, collections, glob

arq = sorted(glob.glob('/content/drive/MyDrive/tcc/artifacts/runs/*.jsonl'))[-1]
linhas = [json.loads(l) for l in open(arq, encoding='utf-8')]
print(arq, '|', len(linhas), 'linhas')

falhas = collections.Counter(r['exec_error'] for r in linhas if not r['exec_ok'])
print('\nerros de execução mais comuns:')
for erro, n in falhas.most_common(5):
    print(f'  {n:>4}  {erro}')

print('\n--- exemplo de saída bruta do modelo ---')
print(linhas[0]['raw_output'][:600])
print('\n--- SQL extraído ---')
print(linhas[0]['generated_sql'])
print('\n--- SQL de referência ---')
print(linhas[0]['gold_sql'])

---## Depois dissoCom a calibração coerente, a ordem é: baselines C1, C2 e C3 → método C4 →varredura de `k`, `n_candidates` e `lambda_question` na validação →congelar a configuração → teste confirmatório com `--split test`.O `run` é retomável: se a sessão cair, rode a mesma célula de novo que elepula o que já está no log.